<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/reactions/natural_gas_combustion_with_cantera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Natural-gas combustion and air blending with Cantera

This engineering example calculates combustion products for natural gases of different compositions while varying the air-to-fuel ratio. It uses [Cantera](https://cantera.org/) and the detailed `gri30.yaml` natural-gas combustion mechanism.

The notebook answers practical screening questions:

* How much stoichiometric air is required by each gas composition?
* How do excess air, fuel richness, hydrogen blending, CO2 dilution, and oxidizer composition affect adiabatic flame temperature?
* What are the wet and dry flue-gas concentrations of CO2, H2O, O2, CO, H2, unburned CH4, and equilibrium NOx?
* How do illustrative boiler, furnace, low-NOx, gas-turbine, duct-burner, and flare cases compare?
* How can Cantera be implemented as a custom unit operation inside a NeqSim process simulation?

The central variable is the excess-air ratio

$$\lambda = \frac{(air/fuel)_{actual}}{(air/fuel)_{stoichiometric}} = \frac{1}{\phi},$$

where $\phi$ is Cantera's equivalence ratio. Thus $\lambda<1$ is fuel-rich, $\lambda=1$ is stoichiometric, and $\lambda>1$ is fuel-lean (excess air).

## Model scope

Cantera provides thermodynamics, chemical kinetics, and transport models for reacting systems. Here we first use **adiabatic, constant-pressure chemical equilibrium** (`equilibrate("HP")`). Enthalpy and pressure are held constant while the product composition and temperature are solved.

This is an excellent first-pass heat-and-material-balance model, but it is not a burner performance guarantee. It assumes complete mixing, no heat loss, and enough time to reach equilibrium. Later sections explain why equilibrium CO and NOx must not be treated as stack-emission predictions.

In [ ]:
# Install only when needed (for example, in a fresh Google Colab runtime).
import importlib.util
import subprocess
import sys

required = []
if importlib.util.find_spec("cantera") is None:
    required.append("cantera>=3.1")
if importlib.util.find_spec("neqsim") is None:
    required.append("neqsim")
if required:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])

In [ ]:
import cantera as ct
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

print(f"Cantera version: {ct.__version__}")
gas_check = ct.Solution("gri30.yaml")
print(f"Mechanism: {gas_check.source}; {gas_check.n_species} species; {gas_check.n_reactions} reactions")

## Fuel definitions

The examples are normalized mole fractions. They deliberately cover lean pipeline gas, a richer gas, CO2-diluted gas, and a 20 mol% hydrogen blend. `gri30.yaml` includes methane, ethane, and propane but is not a general mechanism for arbitrary C4+ natural gas, sulfur species, soot, or ammonia. Use a validated mechanism containing every required fuel and pollutant species for those cases.

The oxidizer below represents dry atmospheric air. Separate oxidizer-blending cases are introduced later.

In [ ]:
def normalize(composition):
    total = sum(composition.values())
    if total <= 0:
        raise ValueError("Composition total must be positive")
    return {name: value / total for name, value in composition.items() if value > 0}


FUELS = {
    "Lean pipeline gas": normalize({"CH4": 0.93, "C2H6": 0.04, "C3H8": 0.01, "CO2": 0.01, "N2": 0.01}),
    "Rich natural gas": normalize({"CH4": 0.82, "C2H6": 0.10, "C3H8": 0.04, "CO2": 0.02, "N2": 0.02}),
    "CO2-diluted gas": normalize({"CH4": 0.88, "CO2": 0.10, "N2": 0.02}),
    "20% H2 blend": normalize({"CH4": 0.76, "H2": 0.20, "C2H6": 0.02, "CO2": 0.01, "N2": 0.01}),
}

DRY_AIR = normalize({"O2": 0.2095, "N2": 0.7808, "AR": 0.0093, "CO2": 0.0004})

fuel_table = pd.DataFrame(FUELS).fillna(0.0).T * 100
fuel_table.index.name = "Fuel [mol%]"
display(fuel_table.round(2))

## Stoichiometric demand and carbon intensity

For one mole of a species $C_xH_yO_z$, complete combustion requires

$$\nu_{O_2}=x+\frac{y}{4}-\frac{z}{2}.$$

The calculation below applies the elemental balance to the complete mixture. The reported CO2 factor is the maximum stack CO2 if all feed carbon exits as CO2. The separate “formed” factor excludes CO2 already present in the fuel. Whether feed CO2 is counted in an emissions inventory depends on its origin and the applicable reporting boundary.

In [ ]:
def fuel_balance_metrics(fuel):
    mechanism = ct.Solution("gri30.yaml")
    fuel = normalize(fuel)
    atoms = {element: 0.0 for element in ("C", "H", "O")}
    molar_mass = 0.0  # kg/kmol
    for species, fraction in fuel.items():
        k = mechanism.species_index(species)
        molar_mass += fraction * mechanism.molecular_weights[k]
        for element in atoms:
            atoms[element] += fraction * mechanism.n_atoms(species, element)

    o2_stoich = atoms["C"] + atoms["H"] / 4.0 - atoms["O"] / 2.0
    feed_co2 = fuel.get("CO2", 0.0)
    mw_co2 = mechanism.molecular_weights[mechanism.species_index("CO2")]
    return {
        "fuel MW [kg/kmol]": molar_mass,
        "O2 stoich [kmol/kmol fuel]": o2_stoich,
        "dry air stoich [kmol/kmol fuel]": o2_stoich / DRY_AIR["O2"],
        "carbon [kmol C/kmol fuel]": atoms["C"],
        "max stack CO2 [kg/kg fuel]": atoms["C"] * mw_co2 / molar_mass,
        "CO2 formed [kg/kg fuel]": (atoms["C"] - feed_co2) * mw_co2 / molar_mass,
    }


fuel_metrics = pd.DataFrame({name: fuel_balance_metrics(comp) for name, comp in FUELS.items()}).T
display(fuel_metrics.round(4))

## Reusable equilibrium-combustion calculation

The function uses Cantera's `set_equivalence_ratio` to blend fuel and oxidizer at a requested $\lambda$, then equilibrates the mixture at constant enthalpy and pressure. Results are reported on both wet and dry bases. Dry ppm values exclude water, matching common flue-gas reporting practice.

The NOx value is the equilibrium sum of NO and NO2. The final column applies the common oxygen-reference conversion to 15 vol% O2 on a dry basis. Confirm the required reference oxygen and regulatory convention before using that conversion in a real report.

In [ ]:
def run_combustion(fuel_name, fuel, lambda_air, oxidizer=DRY_AIR,
                   oxidizer_name="Dry air", inlet_temperature_K=288.15,
                   pressure_bar=1.01325):
    if lambda_air <= 0:
        raise ValueError("lambda_air must be positive")

    gas = ct.Solution("gri30.yaml")
    gas.TP = inlet_temperature_K, pressure_bar * 1e5
    gas.set_equivalence_ratio(
        phi=1.0 / lambda_air,
        fuel=normalize(fuel),
        oxidizer=normalize(oxidizer),
        basis="mole",
    )
    gas.equilibrate("HP")

    x = dict(zip(gas.species_names, gas.X))
    dry_total = max(1.0 - x.get("H2O", 0.0), 1e-30)
    dry = lambda species: x.get(species, 0.0) / dry_total
    o2_dry_pct = 100.0 * dry("O2")
    nox_dry_ppm = 1e6 * (dry("NO") + dry("NO2"))
    correction_15 = ((20.9 - 15.0) / (20.9 - o2_dry_pct)) if o2_dry_pct < 20.9 else np.nan

    return {
        "fuel": fuel_name,
        "oxidizer": oxidizer_name,
        "lambda": lambda_air,
        "excess air [%]": 100.0 * (lambda_air - 1.0),
        "T adiabatic [degC]": gas.T - 273.15,
        "H2O wet [vol%]": 100.0 * x.get("H2O", 0.0),
        "CO2 dry [vol%]": 100.0 * dry("CO2"),
        "O2 dry [vol%]": o2_dry_pct,
        "CO dry [ppmv]": 1e6 * dry("CO"),
        "NOx dry [ppmv]": nox_dry_ppm,
        "NOx dry @15% O2 [ppmv]": nox_dry_ppm * correction_15,
        "CH4 dry [ppmv]": 1e6 * dry("CH4"),
        "H2 dry [vol%]": 100.0 * dry("H2"),
    }


reference = run_combustion("Lean pipeline gas", FUELS["Lean pipeline gas"], lambda_air=1.15)
display(pd.DataFrame([reference]).set_index(["fuel", "oxidizer"]).round(3))

## Sensitivity to fuel composition and air-to-fuel ratio

The sweep spans rich operation ($\lambda=0.8$), stoichiometric operation, and 5–100% excess air. In normal industrial combustion, the useful operating window is constrained by flame stability, CO, NOx, temperature, efficiency, equipment design, and safety systems; it is not selected from equilibrium results alone.

In [ ]:
lambdas = [0.80, 1.00, 1.05, 1.15, 1.30, 1.60, 2.00]
results = pd.DataFrame([
    run_combustion(name, composition, lambda_air)
    for name, composition in FUELS.items()
    for lambda_air in lambdas
])

summary_columns = [
    "fuel", "lambda", "T adiabatic [degC]", "H2O wet [vol%]",
    "CO2 dry [vol%]", "O2 dry [vol%]", "CO dry [ppmv]",
    "NOx dry [ppmv]", "CH4 dry [ppmv]", "H2 dry [vol%]",
]
display(results[summary_columns].round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
plots = [
    ("T adiabatic [degC]", "Adiabatic temperature [degC]", False),
    ("CO2 dry [vol%]", "Dry CO2 [vol%]", False),
    ("O2 dry [vol%]", "Dry O2 [vol%]", False),
    ("CO dry [ppmv]", "Equilibrium dry CO [ppmv]", True),
]

for fuel_name, group in results.groupby("fuel"):
    for ax, (column, ylabel, log_scale) in zip(axes.flat, plots):
        ax.plot(group["lambda"], group[column].clip(lower=1e-6), marker="o", label=fuel_name)
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        if log_scale:
            ax.set_yscale("log")

for ax in axes[-1, :]:
    ax.set_xlabel("Excess-air ratio, lambda [-]")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Fuel composition and air blending: adiabatic equilibrium screening")
fig.tight_layout()
plt.show()

### Reading the trends

* Near-stoichiometric combustion gives the highest adiabatic temperature. Both excess air and fuel-rich operation reduce it.
* More excess air dilutes CO2 and H2O, raises residual O2, and increases the mass/volume of flue gas that equipment must handle.
* Rich operation produces substantial equilibrium CO and H2 because there is insufficient oxygen for complete conversion.
* CO2-diluted fuel has a lower flame temperature because inert material absorbs heat. Its total stack CO2 includes CO2 entering with the fuel.
* Hydrogen blending changes stoichiometric demand, water production, temperature, and CO2 per unit fuel. Compare on an energy basis—not only per kg or per kmol—when assessing decarbonization.
* Equilibrium thermal NO generally rises strongly with flame temperature and available oxygen. Actual NOx is controlled by finite-rate chemistry, residence time, mixing, pressure, heat loss, burner geometry, and staging.

## Changing the oxidizer blend

“Air blending” can mean changing the amount of air, represented by $\lambda$, or changing the oxidizer composition. The cases below compare dry air, humid air, an illustrative 10% exhaust-gas-recirculation (EGR) blend, and oxygen-enriched air at the same $\lambda=1.15$.

The EGR composition is only a screening proxy. In a plant model, recycle composition should be taken from the calculated or measured flue gas and solved iteratively with the burner and recycle loop.

In [ ]:
OXIDIZERS = {
    "Dry air": DRY_AIR,
    "2% humid air": normalize({**{k: 0.98 * v for k, v in DRY_AIR.items()}, "H2O": 0.02}),
    "10% EGR proxy": normalize({
        "O2": 0.90 * DRY_AIR["O2"],
        "N2": 0.90 * DRY_AIR["N2"],
        "AR": 0.90 * DRY_AIR["AR"],
        "CO2": 0.90 * DRY_AIR["CO2"] + 0.07,
        "H2O": 0.03,
    }),
    "25% O2 enriched": normalize({"O2": 0.25, "N2": 0.75}),
}

oxidizer_results = pd.DataFrame([
    run_combustion(
        "Lean pipeline gas", FUELS["Lean pipeline gas"], lambda_air=1.15,
        oxidizer=composition, oxidizer_name=name,
    )
    for name, composition in OXIDIZERS.items()
])

display(oxidizer_results[[
    "oxidizer", "T adiabatic [degC]", "H2O wet [vol%]", "CO2 dry [vol%]",
    "O2 dry [vol%]", "CO dry [ppmv]", "NOx dry [ppmv]",
]].set_index("oxidizer").round(3))

Humid air and EGR add heat capacity and normally lower flame temperature. Oxygen enrichment reduces nitrogen dilution and can strongly increase temperature, so materials limits and burner design become critical. Comparing cases at equal $\lambda$ does not imply equal total oxidizer flow, oxygen concentration, flame speed, or equipment duty.

## Examples for burners, furnaces, gas turbines, duct firing, and flares

Different combustion devices expose the same chemistry to very different mixing, pressure, temperature, residence-time, and heat-transfer conditions. The cases below are transparent **engineering archetypes**, not vendor performance data.

| Device | Important real physics | Useful Cantera model |
|---|---|---|
| Atmospheric or packaged premixed burner | premixing, flame speed, flashback, blowoff | equilibrium screening followed by `FreeFlame` |
| Non-premixed furnace burner | fuel/air jets, recirculation, local stoichiometric surface | staged reactors or `CounterflowDiffusionFlame` |
| Low-NOx staged burner | rich/lean zones, secondary air, internal/external EGR | reactor network with finite-rate kinetics |
| Fired heater or boiler furnace | burner plus radiant/convective heat transfer and leakage air | reactor network coupled to heat-loss or process-duty model |
| Lean-premixed gas turbine | compressor-discharge air, high pressure, dilution, short residence time | high-pressure stirred-reactor network and validated kinetics |
| Diffusion gas-turbine combustor | near-stoichiometric primary zone followed by dilution air | two-zone or multi-zone reactor network |
| Duct burner | hot, oxygen-depleted, water/CO2-rich turbine exhaust | reactor using vitiated oxidizer composition |
| Flare | open turbulent diffusion flame, wind, radiation, incomplete destruction | diffusion-flame or CFD model; equilibrium is only a bound |

The comparison includes separate fuel and oxidizer inlet temperatures. This matters for preheated furnace air and compressor-discharge air in gas turbines.

In [ ]:
def mixture_from_separate_streams(
    fuel, oxidizer, lambda_air, fuel_temperature_K, oxidizer_temperature_K, pressure_bar
):
    '''Mix one kmol of fuel with the requested oxidizer flow at constant pressure.'''
    fuel = normalize(fuel)
    oxidizer = normalize(oxidizer)
    if oxidizer.get("O2", 0.0) <= 0:
        raise ValueError("Oxidizer must contain oxygen")

    pressure_Pa = pressure_bar * 1e5
    mechanism = ct.Solution("gri30.yaml")
    balance = fuel_balance_metrics(fuel)
    oxidizer_kmol = (
        lambda_air * balance["O2 stoich [kmol/kmol fuel]"] / oxidizer["O2"]
    )

    fuel_phase = ct.Solution("gri30.yaml")
    fuel_phase.TPX = fuel_temperature_K, pressure_Pa, fuel
    oxidizer_phase = ct.Solution("gri30.yaml")
    oxidizer_phase.TPX = oxidizer_temperature_K, pressure_Pa, oxidizer

    fuel_mass = fuel_phase.mean_molecular_weight  # kg on the 1 kmol fuel basis
    oxidizer_mass = oxidizer_kmol * oxidizer_phase.mean_molecular_weight
    total_mass = fuel_mass + oxidizer_mass
    total_enthalpy = (
        fuel_phase.enthalpy_mole + oxidizer_kmol * oxidizer_phase.enthalpy_mole
    )

    mixed_moles = dict(fuel)
    for species, fraction in oxidizer.items():
        mixed_moles[species] = mixed_moles.get(species, 0.0) + oxidizer_kmol * fraction

    mechanism.TPX = 300.0, pressure_Pa, mixed_moles
    mechanism.HP = total_enthalpy / total_mass, pressure_Pa
    return mechanism, {
        "fuel mass [kg/basis]": fuel_mass,
        "oxidizer mass [kg/basis]": oxidizer_mass,
        "mixture mass [kg/basis]": total_mass,
        "oxidizer/fuel [kg/kg]": oxidizer_mass / fuel_mass,
        "oxidizer [kmol/kmol fuel]": oxidizer_kmol,
    }


def equilibrium_product_metrics(gas):
    x = dict(zip(gas.species_names, gas.X))
    dry_total = max(1.0 - x.get("H2O", 0.0), 1e-30)
    dry = lambda species: x.get(species, 0.0) / dry_total
    return {
        "T adiabatic [degC]": gas.T - 273.15,
        "H2O wet [vol%]": 100.0 * x.get("H2O", 0.0),
        "CO2 dry [vol%]": 100.0 * dry("CO2"),
        "O2 dry [vol%]": 100.0 * dry("O2"),
        "CO dry [ppmv]": 1e6 * dry("CO"),
        "NOx dry [ppmv]": 1e6 * (dry("NO") + dry("NO2")),
        "H2 dry [vol%]": 100.0 * dry("H2"),
    }


def run_device_case(case):
    gas, basis = mixture_from_separate_streams(
        case["fuel"], case["oxidizer"], case["lambda"],
        case["fuel T [K]"], case["oxidizer T [K]"], case["pressure [bara]"],
    )
    gas.equilibrate("HP")
    row = {
        "category": case["category"],
        "device": case["device"],
        "fuel case": case["fuel case"],
        "lambda": case["lambda"],
        "pressure [bara]": case["pressure [bara]"],
        "oxidizer T [degC]": case["oxidizer T [K]"] - 273.15,
        "oxidizer/fuel [kg/kg]": basis["oxidizer/fuel [kg/kg]"],
        **equilibrium_product_metrics(gas),
    }

    target_C = case.get("process outlet [degC]")
    row["process outlet [degC]"] = target_C
    row["sensible heat recovered [MJ/kg fuel]"] = np.nan
    if target_C is not None:
        cooled = ct.Solution("gri30.yaml")
        cooled.TPY = target_C + 273.15, case["pressure [bara]"] * 1e5, gas.Y
        recovered = (
            (gas.enthalpy_mass - cooled.enthalpy_mass)
            * basis["mixture mass [kg/basis]"]
            / basis["fuel mass [kg/basis]"]
            / 1e6
        )
        row["sensible heat recovered [MJ/kg fuel]"] = recovered
    return row

In [ ]:
EGR_OXIDIZER = OXIDIZERS["10% EGR proxy"]
VITIATED_TURBINE_EXHAUST = normalize({
    "O2": 0.13, "N2": 0.735, "CO2": 0.045, "H2O": 0.08, "AR": 0.01,
})

DEVICE_CASES = [
    {
        "category": "Boiler burner", "device": "Atmospheric premixed boiler burner",
        "fuel case": "Lean pipeline gas", "fuel": FUELS["Lean pipeline gas"],
        "oxidizer": DRY_AIR, "lambda": 1.25, "fuel T [K]": 288.15,
        "oxidizer T [K]": 288.15, "pressure [bara]": 1.01325,
        "process outlet [degC]": 180.0,
    },
    {
        "category": "Fired heater", "device": "Natural-draft furnace burner",
        "fuel case": "Rich natural gas", "fuel": FUELS["Rich natural gas"],
        "oxidizer": DRY_AIR, "lambda": 1.15, "fuel T [K]": 298.15,
        "oxidizer T [K]": 298.15, "pressure [bara]": 1.02,
        "process outlet [degC]": 300.0,
    },
    {
        "category": "Process furnace", "device": "Forced-draft burner with air preheat",
        "fuel case": "Lean pipeline gas", "fuel": FUELS["Lean pipeline gas"],
        "oxidizer": DRY_AIR, "lambda": 1.08, "fuel T [K]": 323.15,
        "oxidizer T [K]": 550.0, "pressure [bara]": 1.05,
        "process outlet [degC]": 900.0,
    },
    {
        "category": "Low-NOx furnace", "device": "Staged burner with 10% EGR proxy",
        "fuel case": "Lean pipeline gas", "fuel": FUELS["Lean pipeline gas"],
        "oxidizer": EGR_OXIDIZER, "lambda": 1.10, "fuel T [K]": 323.15,
        "oxidizer T [K]": 473.15, "pressure [bara]": 1.05,
        "process outlet [degC]": 900.0,
    },
    {
        "category": "Industrial gas turbine", "device": "Lean-premixed combustor",
        "fuel case": "Lean pipeline gas", "fuel": FUELS["Lean pipeline gas"],
        "oxidizer": DRY_AIR, "lambda": 2.40, "fuel T [K]": 323.15,
        "oxidizer T [K]": 700.0, "pressure [bara]": 18.0,
    },
    {
        "category": "Aeroderivative gas turbine", "device": "Lean-premixed H2-blend combustor",
        "fuel case": "20% H2 blend", "fuel": FUELS["20% H2 blend"],
        "oxidizer": DRY_AIR, "lambda": 2.80, "fuel T [K]": 323.15,
        "oxidizer T [K]": 750.0, "pressure [bara]": 30.0,
    },
    {
        "category": "Duct burner", "device": "Supplementary firing in turbine exhaust",
        "fuel case": "Lean pipeline gas", "fuel": FUELS["Lean pipeline gas"],
        "oxidizer": VITIATED_TURBINE_EXHAUST, "lambda": 1.50,
        "fuel T [K]": 298.15, "oxidizer T [K]": 750.0,
        "pressure [bara]": 1.05,
    },
    {
        "category": "Flare", "device": "Open diffusion-flame equilibrium bound",
        "fuel case": "Rich natural gas", "fuel": FUELS["Rich natural gas"],
        "oxidizer": DRY_AIR, "lambda": 1.50, "fuel T [K]": 288.15,
        "oxidizer T [K]": 288.15, "pressure [bara]": 1.01325,
    },
]

device_results = pd.DataFrame([run_device_case(case) for case in DEVICE_CASES])
display(device_results[[
    "category", "device", "fuel case", "lambda", "pressure [bara]",
    "oxidizer T [degC]", "oxidizer/fuel [kg/kg]", "T adiabatic [degC]",
    "CO2 dry [vol%]", "O2 dry [vol%]", "CO dry [ppmv]",
    "NOx dry [ppmv]", "process outlet [degC]",
    "sensible heat recovered [MJ/kg fuel]",
]].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_data = device_results.set_index("device")
plot_data["T adiabatic [degC]"].sort_values().plot.barh(ax=axes[0], color="firebrick")
axes[0].set_xlabel("Adiabatic equilibrium temperature [degC]")
axes[0].set_ylabel("")
axes[0].grid(True, axis="x", alpha=0.3)

plot_data["O2 dry [vol%]"].sort_values().plot.barh(ax=axes[1], color="steelblue")
axes[1].set_xlabel("Dry product O2 [vol%]")
axes[1].set_ylabel("")
axes[1].grid(True, axis="x", alpha=0.3)
fig.suptitle("Illustrative combustion-device operating cases")
fig.tight_layout()
plt.show()

The “sensible heat recovered” column cools the frozen equilibrium products to an illustrative furnace or stack temperature. It is not a complete furnace efficiency calculation: radiation, convection, casing losses, incomplete combustion, fuel heating value basis, air leakage, wall temperature, and process-side heat transfer must also be closed.

For gas turbines, the adiabatic result approximates a combustor-exit/turbine-inlet screening temperature for the selected overall air ratio. A real combustor distributes compressor air among primary combustion, liner cooling, dilution, and turbine cooling; the local flame can therefore be much hotter than the mixed turbine-inlet gas.

## Two-zone staged-burner and gas-turbine model

This model first burns with the primary-zone air, then mixes and reacts with secondary/dilution air. If both zones are allowed to reach equilibrium, the final state is close to a one-zone equilibrium result, but the model also exposes the **primary-zone peak temperature**. That peak is important for thermal NO formation, liner/material loading, and the rationale for staging and EGR.

In [ ]:
def run_two_zone_case(name, fuel, oxidizer, lambda_primary, lambda_overall,
                      fuel_temperature_K, oxidizer_temperature_K, pressure_bar):
    if lambda_overall < lambda_primary:
        raise ValueError("Overall lambda must be at least the primary-zone lambda")

    primary, primary_basis = mixture_from_separate_streams(
        fuel, oxidizer, lambda_primary, fuel_temperature_K,
        oxidizer_temperature_K, pressure_bar,
    )
    primary.equilibrate("HP")
    primary_metrics = equilibrium_product_metrics(primary)

    fuel_balance = fuel_balance_metrics(fuel)
    secondary_kmol = (
        (lambda_overall - lambda_primary)
        * fuel_balance["O2 stoich [kmol/kmol fuel]"]
        / normalize(oxidizer)["O2"]
    )
    secondary_air = ct.Solution("gri30.yaml")
    secondary_air.TPX = oxidizer_temperature_K, pressure_bar * 1e5, normalize(oxidizer)
    secondary_mass = secondary_kmol * secondary_air.mean_molecular_weight
    final_mass = primary_basis["mixture mass [kg/basis]"] + secondary_mass
    final_h = (
        primary_basis["mixture mass [kg/basis]"] * primary.enthalpy_mass
        + secondary_mass * secondary_air.enthalpy_mass
    ) / final_mass
    final_y = (
        primary_basis["mixture mass [kg/basis]"] * primary.Y
        + secondary_mass * secondary_air.Y
    ) / final_mass

    final = ct.Solution("gri30.yaml")
    final.TPY = 300.0, pressure_bar * 1e5, final_y
    final.HP = final_h, pressure_bar * 1e5
    final.equilibrate("HP")
    final_metrics = equilibrium_product_metrics(final)
    return {
        "case": name,
        "primary lambda": lambda_primary,
        "overall lambda": lambda_overall,
        "primary T [degC]": primary_metrics["T adiabatic [degC]"],
        "final mixed T [degC]": final_metrics["T adiabatic [degC]"],
        "primary CO [ppmv dry]": primary_metrics["CO dry [ppmv]"],
        "final CO [ppmv dry]": final_metrics["CO dry [ppmv]"],
        "primary NOx [ppmv dry]": primary_metrics["NOx dry [ppmv]"],
        "final NOx [ppmv dry]": final_metrics["NOx dry [ppmv]"],
        "final O2 [vol% dry]": final_metrics["O2 dry [vol%]"],
    }


two_zone_results = pd.DataFrame([
    run_two_zone_case(
        "Rich-primary staged furnace burner", FUELS["Lean pipeline gas"], DRY_AIR,
        lambda_primary=0.85, lambda_overall=1.10,
        fuel_temperature_K=323.15, oxidizer_temperature_K=473.15, pressure_bar=1.05,
    ),
    run_two_zone_case(
        "EGR low-NOx furnace burner", FUELS["Lean pipeline gas"], EGR_OXIDIZER,
        lambda_primary=0.90, lambda_overall=1.15,
        fuel_temperature_K=323.15, oxidizer_temperature_K=473.15, pressure_bar=1.05,
    ),
    run_two_zone_case(
        "Diffusion gas-turbine combustor", FUELS["Lean pipeline gas"], DRY_AIR,
        lambda_primary=1.00, lambda_overall=2.80,
        fuel_temperature_K=323.15, oxidizer_temperature_K=700.0, pressure_bar=20.0,
    ),
])
display(two_zone_results.round(2))

The equilibrium NOx columns illustrate thermodynamic tendency only. In staged systems the species do not re-equilibrate instantaneously: finite residence time, radical chemistry, mixing rate, flame cooling, and quenching control the actual result. A design study should replace each equilibrium zone with a finite-rate Cantera reactor and use measured or CFD-derived residence-time and mixing distributions.

## Bringing a NeqSim composition into Cantera

NeqSim is well suited to upstream PVT and process simulation: separation, compression, fuel-gas conditioning, condensation, and stream properties. Cantera then adds detailed reacting-system calculations. A practical workflow is:

1. Run the fuel-gas conditioning/process case in NeqSim.
2. Extract the gas-phase mole fractions at the burner boundary.
3. Map component names and retain only species supported by the selected Cantera mechanism.
4. Normalize, document any C4+ lumping, and run the combustion sensitivity.
5. Return flue-gas flow, composition, temperature, and heat duty to the wider process/emissions study.

The helper below demonstrates the name mapping. It fails visibly for unsupported nonzero species rather than silently dropping carbon.

In [ ]:
NEQSIM_TO_CANTERA = {
    "methane": "CH4", "ethane": "C2H6", "propane": "C3H8",
    "hydrogen": "H2", "CO2": "CO2", "nitrogen": "N2",
    "oxygen": "O2", "water": "H2O",
}


def map_neqsim_gas_to_cantera(neqsim_mole_fractions, tolerance=1e-12):
    mapped = {}
    unsupported = {}
    for name, fraction in neqsim_mole_fractions.items():
        if fraction <= tolerance:
            continue
        cantera_name = NEQSIM_TO_CANTERA.get(name)
        if cantera_name is None:
            unsupported[name] = fraction
        else:
            mapped[cantera_name] = mapped.get(cantera_name, 0.0) + fraction
    if unsupported:
        raise ValueError(
            "Unsupported nonzero species; choose a suitable mechanism or an explicit lumping rule: "
            + str(unsupported)
        )
    return normalize(mapped)


neqsim_example = {
    "methane": 0.93, "ethane": 0.04, "propane": 0.01,
    "CO2": 0.01, "nitrogen": 0.01,
}
mapped_fuel = map_neqsim_gas_to_cantera(neqsim_example)
display(pd.Series(mapped_fuel, name="Cantera mole fraction").to_frame())

## Wiring Cantera into a NeqSim process simulation

The following example makes Cantera a Python-defined unit operation inside a NeqSim `ProcessSystem`:

**NeqSim fuel feed → pressure-control valve → fuel preheater → Cantera combustor → NeqSim waste-heat cooler**

The interface passes these quantities:

| Direction | Data |
|---|---|
| NeqSim to Cantera | fuel composition, fuel mass flow, fuel temperature, burner pressure |
| Combustor specification | oxidizer composition and temperature, excess-air ratio |
| Cantera to NeqSim | major flue-gas composition, adiabatic temperature, total fuel-plus-oxidizer mass flow |
| Cantera metadata | equilibrium CO, NO, NO2, H2, wet/dry reporting values, omitted trace fraction |

Cantera supplies the chemical reaction and heat release. The returned NeqSim stream is then suitable for downstream sensible cooling, heat recovery, water-condensation screening, piping, or CO2-removal studies. Do not add the combustion heat again in NeqSim—the Cantera outlet temperature already reflects it.

In [ ]:
try:
    from neqsim import jneqsim
except ImportError:
    from neqsim import jNeqSim as jneqsim
from neqsim.process.unitop import unitop
from neqsim.thermo import TPflash

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
Heater = jneqsim.process.equipment.heatexchanger.Heater
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
ProcessSystem = jneqsim.process.processmodel.ProcessSystem


def composition_from_neqsim_stream(stream):
    fluid = stream.getFluid()
    composition = {}
    for i in range(int(fluid.getNumberOfComponents())):
        component = fluid.getComponent(i)
        composition[str(component.getName())] = float(component.getz())
    return normalize(composition)

In [ ]:
CANTERA_TO_NEQSIM_MAJOR = {
    "N2": "nitrogen", "O2": "oxygen", "CO2": "CO2", "H2O": "water",
    "CO": "CO", "H2": "hydrogen", "CH4": "methane", "AR": "argon",
    "NO": "NO", "NO2": "NO2",
}


class CanteraCombustor(unitop):
    '''Steady external combustion unit for a NeqSim ProcessSystem.

    Cantera calculates adiabatic equilibrium. A major-species product stream is
    returned to NeqSim; detailed trace results remain available as metadata.
    '''

    def __init__(self, name, lambda_air=1.15, oxidizer=DRY_AIR,
                 oxidizer_temperature_K=298.15,
                 neqsim_handoff_temperature_K=None):
        super().__init__()
        self.setName(name)
        self.lambda_air = float(lambda_air)
        self.oxidizer = normalize(oxidizer)
        self.oxidizer_temperature_K = float(oxidizer_temperature_K)
        self.neqsim_handoff_temperature_K = neqsim_handoff_temperature_K
        self.inlet = None
        self.outlet = None
        self.last_results = {}

    def setInputStream(self, stream):
        self.inlet = stream
        self.outlet = Stream(f"{self.getName()} outlet", stream.getFluid().clone())
        # Downstream NeqSim units clone this stream during construction. Set the
        # anticipated fuel-plus-oxidizer flow before those units are created.
        cantera_fuel = map_neqsim_gas_to_cantera(composition_from_neqsim_stream(stream))
        _, basis = mixture_from_separate_streams(
            cantera_fuel, self.oxidizer, self.lambda_air,
            float(stream.getTemperature("K")), self.oxidizer_temperature_K,
            float(stream.getPressure("bara")),
        )
        anticipated_flow = (
            float(stream.getFlowRate("kg/hr"))
            * basis["mixture mass [kg/basis]"]
            / basis["fuel mass [kg/basis]"]
        )
        self.outlet.setFlowRate(anticipated_flow, "kg/hr")

    def getOutletStream(self):
        return self.outlet

    def setLambda(self, value):
        self.lambda_air = float(value)

    def getResults(self):
        return dict(self.last_results)

    def run(self, uuid=None):
        if self.inlet is None:
            raise RuntimeError("Set the combustor input stream before running")

        neqsim_composition = composition_from_neqsim_stream(self.inlet)
        cantera_fuel = map_neqsim_gas_to_cantera(neqsim_composition)
        pressure_bar = float(self.inlet.getPressure("bara"))
        fuel_temperature_K = float(self.inlet.getTemperature("K"))
        fuel_flow_kg_per_hr = float(self.inlet.getFlowRate("kg/hr"))

        products, basis = mixture_from_separate_streams(
            cantera_fuel, self.oxidizer, self.lambda_air,
            fuel_temperature_K, self.oxidizer_temperature_K, pressure_bar,
        )
        products.equilibrate("HP")

        handoff_temperature_K = (
            products.T if self.neqsim_handoff_temperature_K is None
            else float(self.neqsim_handoff_temperature_K)
        )
        handoff_products = ct.Solution("gri30.yaml")
        handoff_products.TPY = handoff_temperature_K, pressure_bar * 1e5, products.Y
        handoff_products.equilibrate("TP")

        cantera_x = dict(zip(handoff_products.species_names, handoff_products.X))
        major = {
            CANTERA_TO_NEQSIM_MAJOR[name]: cantera_x.get(name, 0.0)
            for name in CANTERA_TO_NEQSIM_MAJOR
            if cantera_x.get(name, 0.0) > 1e-12
        }
        major_total = sum(major.values())
        omitted_fraction = max(0.0, 1.0 - major_total)
        major = {name: value / major_total for name, value in major.items()}

        product_fluid = SystemSrkEos(handoff_temperature_K, pressure_bar)
        for name, fraction in major.items():
            product_fluid.addComponent(name, fraction)
        product_fluid.setMixingRule("classic")
        TPflash(product_fluid)
        product_fluid.initProperties()

        flue_flow_kg_per_hr = (
            fuel_flow_kg_per_hr
            * basis["mixture mass [kg/basis]"]
            / basis["fuel mass [kg/basis]"]
        )
        self.outlet.setThermoSystem(product_fluid)
        self.outlet.setFlowRate(flue_flow_kg_per_hr, "kg/hr")
        self.outlet.setPressure(pressure_bar, "bara")
        self.outlet.setTemperature(handoff_temperature_K, "K")

        reference_cool_products = ct.Solution("gri30.yaml")
        reference_cool_products.TPY = (
            273.15 + 180.0, pressure_bar * 1e5, handoff_products.Y
        )
        cantera_heat_to_handoff_MW = (
            (products.enthalpy_mass - handoff_products.enthalpy_mass)
            * flue_flow_kg_per_hr / 3.6e9
        )
        cantera_heat_handoff_to_180C_MW = (
            (handoff_products.enthalpy_mass - reference_cool_products.enthalpy_mass)
            * flue_flow_kg_per_hr / 3.6e9
        )

        self.last_results = {
            **equilibrium_product_metrics(products),
            "lambda": self.lambda_air,
            "fuel flow [kg/hr]": fuel_flow_kg_per_hr,
            "oxidizer flow [kg/hr]": flue_flow_kg_per_hr - fuel_flow_kg_per_hr,
            "flue gas flow [kg/hr]": flue_flow_kg_per_hr,
            "oxidizer/fuel [kg/kg]": basis["oxidizer/fuel [kg/kg]"],
            "NeqSim handoff temperature [degC]": handoff_temperature_K - 273.15,
            "Cantera heat removed before handoff [MW]": cantera_heat_to_handoff_MW,
            "Cantera heat handoff-to-180C [MW]": cantera_heat_handoff_to_180C_MW,
            "Cantera species omitted from NeqSim stream [mol fraction]": omitted_fraction,
        }

In [ ]:
# 1) NeqSim fuel-gas feed and conditioning section
fuel_fluid = SystemSrkEos(288.15, 5.0)
fuel_fluid.addComponent("methane", 0.93)
fuel_fluid.addComponent("ethane", 0.04)
fuel_fluid.addComponent("propane", 0.01)
fuel_fluid.addComponent("CO2", 0.01)
fuel_fluid.addComponent("nitrogen", 0.01)
fuel_fluid.setMixingRule("classic")
TPflash(fuel_fluid)

fuel_feed = Stream("Fuel gas feed", fuel_fluid)
fuel_feed.setFlowRate(1000.0, "kg/hr")

fuel_valve = ThrottlingValve("Fuel pressure-control valve", fuel_feed)
fuel_valve.setOutletPressure(1.05)

fuel_preheater = Heater("Fuel preheater", fuel_valve.getOutletStream())
fuel_preheater.setOutTemperature(323.15)

# 2) External Cantera chemistry unit inside the NeqSim flowsheet
combustor = CanteraCombustor(
    "Cantera furnace combustor",
    lambda_air=1.15,
    oxidizer=DRY_AIR,
    oxidizer_temperature_K=298.15,
    neqsim_handoff_temperature_K=273.15 + 900.0,
)
combustor.setInputStream(fuel_preheater.getOutletStream())

# 3) Continue in NeqSim with flue-gas heat recovery
waste_heat_cooler = Cooler("Waste-heat recovery cooler", combustor.getOutletStream())
waste_heat_cooler.setOutTemperature(273.15 + 180.0)

integrated_process = ProcessSystem()
for unit in [fuel_feed, fuel_valve, fuel_preheater, combustor, waste_heat_cooler]:
    integrated_process.add(unit)
integrated_process.run()

bridge_results = combustor.getResults()
bridge_results["NeqSim cooler outlet [degC]"] = float(
    waste_heat_cooler.getOutletStream().getTemperature("C")
)
bridge_results["NeqSim recovered sensible duty [MW]"] = abs(
    float(waste_heat_cooler.getDuty())
) / 1e6
bridge_results["NeqSim vs Cantera cooling-duty difference [%]"] = 100.0 * (
    bridge_results["NeqSim recovered sensible duty [MW]"]
    / bridge_results["Cantera heat handoff-to-180C [MW]"] - 1.0
)
display(pd.DataFrame([bridge_results]).T.rename(columns={0: "value"}).round(4))

### Extending the integrated process

The custom unit exposes `setLambda()`, so a NeqSim process study can sweep burner air demand together with upstream fuel conditioning and downstream heat recovery. Useful extensions include:

* connect fuel composition to a separator, fuel-gas scrubber, membrane, or stabilizer model;
* add air-compressor power and compressor-discharge temperature for a gas turbine;
* iterate a flue-gas recycle stream into the oxidizer definition;
* cool below the water dew point and add a separator for condensate recovery;
* feed dry flue gas to a CO2-capture or compression model;
* wrap the combustor and heat-recovery section in NeqSim design cases or optimization loops.

The returned NeqSim stream intentionally contains major species only. Cantera retains the detailed trace composition in `combustor.getResults()`. For elemental and mass closure stricter than the omitted trace fraction, expand the mapping only after confirming that each species exists and is thermodynamically supported in NeqSim.

At the selected handoff temperature, the notebook equilibrates the Cantera products at constant temperature to remove short-lived radicals before mapping supported species to NeqSim. Trace-emission values are still taken from the hot combustor calculation and must be treated as screening values.

The table deliberately reports the same 900-to-180 °C cooling interval using both Cantera and NeqSim. Differences reveal that the packages use different ideal-gas heat-capacity correlations, reference states, species sets, and high-temperature validity ranges. For a closed combustion energy balance, retain Cantera enthalpy differences through the reacting and high-temperature flue-gas section, reconcile reference states at a documented handoff, and validate the selected downstream property model. Do not silently splice duties from the two packages.

## Engineering use and limitations

This notebook can support concept selection, burner/furnace heat-and-material balances, gas-turbine fuel sensitivity, flue-gas equipment screening, CO2 capture inlet estimates, oxygen-analyzer cross-checks, EGR/oxygen-enrichment studies, and scenario generation for emissions work.

For an engineering study, extend the result object with fuel flow and heating value, calculate wet and dry flue-gas flow rates, close the heat balance against a specified furnace duty or turbine efficiency, and propagate uncertainty in fuel analysis, humidity, leakage air, and measurement error.

Important limitations:

* **Temperature:** adiabatic equilibrium temperature is an upper-bound reference, not a measured flame, turbine-inlet, furnace-exit, or stack temperature.
* **CO and unburned hydrocarbons:** equilibrium does not represent quenching, incomplete mixing, wall effects, flame instability, or finite residence time.
* **NOx:** equilibrium NOx is not a regulatory emissions prediction. Use a validated finite-rate reactor/flame model and burner-specific data for design or compliance work.
* **Mechanism validity:** verify that the mechanism contains and has been validated for the actual fuel species, pressure, temperature, dilution, and equivalence-ratio range.
* **Trace species:** sulfur, particulates, soot, metals, and many fuel contaminants are outside `gri30.yaml`.
* **Basis:** always label wet/dry basis, ppm/vol%, reference O2, pressure, inlet temperature, and whether CO2 already present in the fuel is counted.

## References and next steps

* [Cantera home page](https://cantera.org/)
* [Cantera Python tutorial](https://cantera.org/stable/userguide/python-tutorial.html)
* [Cantera example gallery](https://cantera.org/stable/examples/)

Natural next steps are a constant-pressure stirred-reactor model for residence-time effects, an ignition-delay study, or a one-dimensional premixed-flame model for flame speed and spatial temperature/species profiles.